# Bloque 4 · Elementos de prueba — prácticas en Google Colab
## Del dato a la evidencia verificable

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/consumer-spain/curso-ia-colab/blob/main/bloque4/bloque4_practicas.ipynb)

**Esto es el Plan B.** Si en tu ordenador algo no funciona (Python, paquetes, permisos…), puedes seguir aquí
**todas las prácticas del Bloque 4**, en el mismo orden que en clase.

**Cómo se usa**
- Cada bloque gris es una *celda*. Se ejecuta con el botón ▶ de su izquierda (o **Mayús + Enter**).
- Ejecuta las celdas **en orden**, de arriba abajo.
- Si Colab se desconecta o lo reinicias, vuelve a ejecutar la celda **Preparación**.
- Colab **no guarda tus cambios**. Si quieres conservarlos: *Archivo → Guardar una copia en Drive*.
- No subas aquí datos reales ni información interna: todo el material del curso es de prueba.

| En clase | Aquí |
|---|---|
| Doble clic en un `.bat` o arrastrar archivos encima | Ejecutar una celda (`!python archivo.py …`) |
| Explorador: copiar, pegar y renombrar (Ctrl+C, Ctrl+V, F2) | Una celda que hace lo mismo con Python |
| Bloc de notas, Paint, Excel | Una celda que muestra el archivo (`!cat`, `display`, pandas) o que lo modifica |
| La ventana del sellador de la Bola extra (`tkinter`) | Una celda que lo ejecuta por terminal |
| `/exif`, `/hash`, `/ip`, `/verificador` en la web del curso | **Igual**, en tu navegador. Los archivos se descargan desde Colab |

> **Importante.** Lo que se obtiene desde Colab sale de servidores de Google, no de un equipo institucional:
> **no tiene valor probatorio**. Es una réplica de las prácticas para aprender, no una forma de producir evidencia.

## Preparación

Descarga los materiales del Bloque 4 (el mismo ZIP que en clase), instala lo que falte y ejecuta el comprobador.
Es el equivalente a `COMPROBAR_BLOQUE4.bat`. Debe terminar en **TODO LISTO** o **HAY ALGO QUE REVISAR**.

*Adaptada.* En Colab no hay ventanas (`tkinter` puede estar instalado, pero no tiene dónde dibujar).
El comprobador también hace una prueba de sellado con la TSA: puede tardar hasta 10 segundos.

In [ ]:
# PREPARACIÓN · ejecuta esta celda primero
import os, sys, zipfile, urllib.request, subprocess
from pathlib import Path

ZIP_URL = "https://raw.githubusercontent.com/consumer-spain/curso-ia-colab/main/bloque4/BLOQUE4_ALUMNADO.zip"
TRABAJO = Path.cwd().resolve()
if "BLOQUE4" in TRABAJO.parts:      # si se vuelve a ejecutar esta celda, no se anida otra carpeta BLOQUE4
    TRABAJO = Path(*TRABAJO.parts[:TRABAJO.parts.index("BLOQUE4")])
BASE = TRABAJO / "BLOQUE4"

if not BASE.exists():
    print("Descargando los materiales del Bloque 4...")
    urllib.request.urlretrieve(ZIP_URL, TRABAJO / "materiales_b4.zip")
    with zipfile.ZipFile(TRABAJO / "materiales_b4.zip") as z:
        z.extractall(TRABAJO)
    print("Materiales listos en:", BASE)
else:
    print("Los materiales ya estaban descargados:", BASE)

# Paquetes (normalmente Colab ya los trae)
for modulo, paquete in (("pandas", "pandas"), ("openpyxl", "openpyxl"), ("requests", "requests"),
                        ("bs4", "beautifulsoup4"), ("PIL", "Pillow")):
    try:
        __import__(modulo)
    except ImportError:
        print("Instalando", paquete, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import pandas as pd
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 60)
from IPython.display import Image as VerImagen, display


def ir_a(carpeta):
    """Se coloca en una carpeta del material (como abrir la carpeta en el Explorador)."""
    os.chdir(BASE / carpeta if carpeta else BASE)
    print("Estás en:", Path.cwd().relative_to(TRABAJO))


def ver_excel(ruta, hoja=0):
    """Muestra un Excel (en clase se abre con Excel)."""
    return pd.read_excel(ruta, sheet_name=hoja)


def descargar(*rutas):
    """Descarga archivos a tu ordenador (para usarlos en la web del curso o en tu IA)."""
    try:
        from google.colab import files
        for r in rutas:
            files.download(str(r))
    except ImportError:
        print("Esta celda solo descarga archivos dentro de Google Colab.")


def ultimo_paquete():
    """Devuelve la carpeta EVIDENCIA_… más reciente creada por tu sellador (o None)."""
    base = BASE / "05_BOLA_EXTRA_SELLADOR" / "MIS_PAQUETES"
    if not base.exists():
        return None
    paquetes = sorted(base.glob("EVIDENCIA_*"), key=lambda p: p.stat().st_mtime)
    return paquetes[-1] if paquetes else None


ir_a("")
print()
r = subprocess.run([sys.executable, "comprobar_bloque4.py"], capture_output=True, text=True)
import re
print(re.split(r"\n(?=HAY ALGO QUE REVISAR|TODO LISTO)", r.stdout)[0])
problemas = [l for l in r.stdout.splitlines() if l.startswith("  - ")]
otros = [l for l in problemas if "tkinter" not in l]
TSA_OK = "sello real recibido" in r.stdout
if r.returncode == 0:
    print("TODO LISTO")
elif not otros:
    print("TODO LISTO (en Colab no hay ventanas: la falta de tkinter es esperable)")
else:
    print("HAY ALGO QUE REVISAR")
    for l in otros:
        print(l)
print("\nNota: la prueba de sellado con la TSA solo hace falta para la Bola extra y aquí se omite.")
print("En clase los paquetes ya vienen sellados: las prácticas de VERIFICAR no la necesitan.")

---
# A · Apertura

**Muchas veces, el dato basta.** Estudios de mercado, monitorización, estadística, seguimiento de precios, indicios.
**A veces necesitamos algo más:** poder acreditarlo después.

### La pregunta de hoy
**¿Cómo pasamos de un dato útil a una evidencia que pueda comprobarse después?**

Hoy lo vamos a descubrir con un ejemplo y con herramientas. *No hay nada que ejecutar en esta parte.*

---
# B · Books to Scrape

Partimos de lo que ya sabemos hacer (Bloque 3): obtener una tabla de una web. Aquí, los 20 libros de la primera página
de *Books to Scrape*, una web de pruebas.

> **IGUAL / ADAPTADA.** El programa es el mismo. En clase se lanza con `EJECUTAR_BOOKS.bat`; aquí con una celda.
> Si la web no responde desde los servidores de Google, la celda repite sola con la copia local de respaldo
> (exactamente lo que hace el `.bat`).

## Ejecutar

In [ ]:
ir_a("01_BOOKS")
!python obtener_libros.py || (echo "La web no ha respondido. Repetimos con la copia local de respaldo..." && python obtener_libros.py --respaldo)

## Abrir el Excel

En clase se abre `libros_books.xlsx` con Excel. Aquí lo mostramos con pandas (también puedes descargarlo con el panel 📁).

In [ ]:
ver_excel("libros_books.xlsx")

## ¿Lo daríais por bueno?

Piénsalo antes de seguir: ¿qué has comprobado de esta tabla? ¿Qué no puedes saber solo con mirarla?

### Mini-reto · fila 7 · por parejas

Fíjate **solo en la fila 7**. Dentro de seis meses alguien te pregunta: *«¿cómo sabemos de dónde salió esta fila?»*

**Para una sola fila: ¿qué guardaríais hoy?** Apunta todas tus ideas (en una celda de texto, en papel…).

*Después, en común, ordenaremos lo que habéis dicho: la página tal como se vio · la URL, la hora y con qué herramienta ·
algo que avise si el archivo cambia · una hora que no dependa de mi equipo · una nota que explique cada archivo · comprobar que todo encaja.*

### Lo que habéis propuesto ya es un método en seis pasos
**RECOGER → CONTEXTUALIZAR → FIJAR → VINCULAR AL TIEMPO → DOCUMENTAR → VERIFICAR**

---
# C · RECOGER

**Guardar la fila no basta: guardamos lo que vimos.** Recogemos la ficha completa del libro 7, tal como llegó,
más un registro de qué, de dónde, cuándo y cómo.

> **IGUAL.** (`RECOGER_LIBRO_7.bat` → una celda; con el mismo respaldo si no hay web.)

In [ ]:
ir_a("01_BOOKS")
!python recoger_libro.py 7 || (echo "Repetimos con la copia local de respaldo..." && python recoger_libro.py 7 --respaldo)

Mira qué se ha creado y lee el registro. Fíjate en los cuatro bloques: **qué se recogió · de dónde · cuándo · cómo**.

In [ ]:
!ls RECOGIDA_libro_07
!cat RECOGIDA_libro_07/registro_recogida.json

> Ojo con el «cuándo»: es la **hora del reloj de este equipo**. Es una declaración, no una garantía externa.

**Recoger = conservar lo que vimos, no solo el dato que extrajimos.**

## Caso guiado · Orion 3

Para los siguientes pasos trabajamos con un caso: la captura de una oferta de la cafetera *Orion 3*,
hecha con *InspectorApp* (una aplicación ficticia del curso). En clase está en `02_INSPECTORAPP`; aquí la ves directamente.

In [ ]:
ir_a("02_INSPECTORAPP")
display(VerImagen("captura_original.jpg", width=760))

---
# D · CONTEXTUALIZAR

**Un archivo no solo contiene lo que vemos.** Los metadatos dicen cosas del archivo sobre sí mismo.

> **Windows → Propiedades → Detalles** (clic derecho sobre la imagen): **No exportable**; en Colab no hay Explorador.
> Usamos directamente la herramienta del curso, que además compara dos archivos.

## Práctica · metadatos de la original y de la reducida

**No leas todo. Busca estos campos:** Software · Dimensiones · Fecha declarada · Dispositivo · GPS.
Primero en `captura_original.jpg`; después compara con `captura_reducida.jpg`.

*(En clase: `1_VER_METADATOS.bat`.)*

In [ ]:
ir_a("02_INSPECTORAPP")
!python ver_metadatos.py captura_original.jpg captura_reducida.jpg

### En la web del curso · `/exif`

*(Igual, en tu navegador.)* Abre `/exif` en la web del curso (la dirección que se proyecta en clase) y sube
`captura_original.jpg` como imagen A y `captura_reducida.jpg` como imagen B. Primero las descargas:

In [ ]:
descargar("captura_original.jpg", "captura_reducida.jpg")

## ¿Qué observas?

- ¿Qué cambia entre las dos? ¿Qué campo sigue siendo el mismo?
- ¿Cuánto puedes fiarte de lo que el propio archivo dice de sí mismo?

*Los metadatos forman parte del archivo: se pueden alterar o desaparecer.* **METADATO ≠ DATO CERTIFICADO.**

### Qué afirmar y qué no
Un metadato permite decir que el archivo **es compatible** con algo, no que ese algo esté demostrado. **COMPATIBLE ≠ DEMOSTRADO.**

## Mini-reto · `/ip` · ¿desde dónde llegas tú?

> **IGUAL, y solo en tu navegador.** Abre `/ip` en la web del curso (QR o dirección proyectada).
> La web te muestra la IP desde la que le llega *tu* petición. **No lo hagas desde Colab:** aquí la petición sale de Google, no de ti.

Pensad: ¿cambia si cambias de red (wifi o datos del móvil)? ¿Una IP identifica a una persona? **IP OBSERVADA ≠ PERSONA IDENTIFICADA.**
Una IP sola frente al registro que guarda el servidor… ya lo hemos comentado: la IP dice desde qué red llegó la petición, no quién la hizo.

*Solo como contraste (ADAPTADA), esta celda pregunta qué IP vería un servidor si la petición saliera de este cuaderno.
Fíjate en que no es la tuya, sino la de un centro de datos de Google:*

In [ ]:
import urllib.request
try:
    ip = urllib.request.urlopen("https://api.ipify.org", timeout=8).read().decode()
    print("Un servidor que recibiera una petición desde este cuaderno vería la IP:", ip)
    print("Es una IP de centro de datos, no la de tu equipo.")
except Exception as exc:
    print("No se ha podido consultar la IP:", exc)

---
# E · FIJAR

**Antes de tocar archivos: ver la extensión.** En clase se activa en Windows *Ver → Extensiones de nombre de archivo*.
*(No exportable: en Colab las extensiones se ven siempre en el panel 📁 y en `!ls`.)*

## Crear una huella del archivo

**ARCHIVO → FUNCIÓN HASH → HUELLA.** Existen MD5, SHA-1, SHA-256 y SHA-512; **en este curso usamos SHA-256**.

## Práctica · ¿Cuáles son el mismo archivo?

**Primero predice. Después compruébalo.** Vamos a crear una copia de `captura_original.jpg` con otro nombre (`mi_copia.jpg`).
¿Qué archivos quedarán en el mismo grupo?

- `captura_original.jpg`
- `mi_copia.jpg`
- `captura_reducida.jpg`

*A la vista, las tres muestran la misma oferta.* **Escribe tu predicción antes de ejecutar nada.**

*(En clase: Ctrl+C → Ctrl+V → F2. Aquí, esta celda: ADAPTADA.)*

In [ ]:
import shutil
ir_a("02_INSPECTORAPP")
shutil.copy("captura_original.jpg", "mi_copia.jpg")
print("Creada mi_copia.jpg")

Ahora calcula las huellas y agrúpalas (en clase: `2_COMPARAR_HASHES.bat`):

In [ ]:
!python comparar_hashes.py captura_original.jpg captura_reducida.jpg mi_copia.jpg

¿Acertaste? ¿Qué te dice esto sobre «el mismo archivo»?

### En la web del curso · `/hash`
*(Igual, en tu navegador.)* Selecciona los tres archivos en `/hash`. Si los necesitas en tu ordenador:

In [ ]:
descargar("captura_original.jpg", "captura_reducida.jpg", "mi_copia.jpg")

## Mini-reto · Cambia un solo carácter

**¿Cuánto cambiará la huella?**

- **A.** cambia un carácter de la huella
- **B.** cambian unos pocos
- **C.** cambia casi todo

Elige antes de ejecutar. Ahora haremos una copia de `ARCHIVO_DE_MUESTRA.txt`, le añadiremos **un punto** al final
y volveremos a calcular la huella. **Se modifica la copia; el original no se toca.**

*(En clase: copiar, Bloc de notas, Ctrl+S, `2_COMPARAR_HASHES.bat`. Aquí: ADAPTADA, con celdas.)*

In [ ]:
import shutil
ir_a("02_INSPECTORAPP")
shutil.copy("ARCHIVO_DE_MUESTRA.txt", "ARCHIVO_DE_MUESTRA - copia.txt")
with open("ARCHIVO_DE_MUESTRA - copia.txt", "a", encoding="utf-8") as f:
    f.write(".")            # un único carácter más
print("Copia modificada: se ha añadido un punto al final.")

In [ ]:
!python comparar_hashes.py "ARCHIVO_DE_MUESTRA.txt" "ARCHIVO_DE_MUESTRA - copia.txt"

Compara las dos huellas, carácter a carácter. ¿Era A, B o C?

## Lo que nos da SHA-256… y lo que no

Nos da una huella que **cambia si cambian los bytes**. No nos dice si el contenido es verdadero.
**INTEGRIDAD ≠ VERACIDAD.**

---
# F · VINCULAR AL TIEMPO

La huella dice *qué* archivo. No dice *cuándo*. Y las tres fechas que tenemos bajo nuestro control
(la del reloj del equipo, la de modificación y la de los metadatos) **se pueden cambiar**.

**Firma, sello y sello de tiempo no son lo mismo.** Un **sello de tiempo** (RFC 3161) lo emite un tercero, una **TSA**:
recibe solo la **huella** del archivo (no el archivo) y devuelve una respuesta firmada con la hora.
Esa respuesta es el `.tsr`; lo que se envía, la petición, es el `.tsq`.

*(Apunte eIDAS: que un sello cumpla RFC 3161 **no** lo hace cualificado; el prestador y el servicio deben estar en la lista de confianza.)*

## Observa una petición real

Demostración: qué viajaría a la TSA, **sin enviar nada**. La petición (`.tsq`) contiene la huella SHA-256; el archivo no sale.

In [ ]:
ir_a("03_PAQUETE_VERIFICACION")
!python ver_peticion.py

## Mini-reto · Marca SÍ o NO

Tenemos un sello de tiempo de `captura_original.jpg`. Responde antes de seguir:

1. ¿Demuestra **quién hizo** la captura?
2. ¿Demuestra que **el precio era verdadero**?
3. ¿Demuestra que **la captura se hizo exactamente en ese instante**?
4. ¿Demuestra que **esos bytes ya existían** cuando se emitió el sello?
5. Si hoy la huella coincide, ¿**seguimos teniendo los mismos bytes**?

<details><summary>Ver las respuestas (después de contestar)</summary>

1. **NO.** 2. **NO.** 3. **NO** (el sello dice «como muy tarde»). 4. **SÍ.** 5. **SÍ.**

**SELLO DE TIEMPO ≠ FECHA DE CREACIÓN.** *El sellado protege hacia adelante; no convierte en verdadera una captura.*
</details>

## Huella del archivo = huella sellada

Para verificar, se comparan **dos huellas**: la que quedó sellada y la del archivo que tienes delante.
Si coinciden, ese archivo es el que se selló. Si no, no lo es.

## Práctica · verifica un sello real, capa por capa

Paquete de la cafetera **Orion 3** (paquete B), sellado con la TSA de la ACCV. Lee **cada lectura por separado**:
A · la TSA concedió el sello · B · origen/TSA · C · instante que declara la TSA · D · huella que se selló · E · ¿corresponde a ESTE archivo?

**Aquí, sin web** (en clase: `3_VERIFICAR_SELLO.bat`). Este programa **no comprueba la firma ni la cadena de confianza**:

In [ ]:
ir_a("03_PAQUETE_VERIFICACION/EVIDENCIA_orion3_paquete_B")
!python ../../02_INSPECTORAPP/verificar_sello.py original.jpg sello.tsr

**En la web del curso · `/verificador`** *(Igual, en tu navegador. Allí sí se comprueban la firma y la cadena.)*
Descarga los dos archivos y súbelos:

In [ ]:
ir_a("03_PAQUETE_VERIFICACION/EVIDENCIA_orion3_paquete_B")
descargar("original.jpg", "sello.tsr")

## El momento de fijación

Lo que se sella es la **huella del archivo tal como existe en ese instante**. Si se sella una captura ya manipulada,
el sello certifica esa manipulación. Por eso el sellado debe ocurrir **en el momento de fijar**, no después.

---
# G · DOCUMENTAR

**Una carpeta, seis meses después.** Alguien abre una carpeta con archivos que se llaman casi igual. ¿Cuál es el original?
¿Cuál es una copia? ¿Qué hace cada uno? **Documentar no es un formato**: es dejar escrito qué es cada cosa y de dónde sale.

## Mini-reto · Original, copia, derivado o modificado

Con los archivos de `02_INSPECTORAPP`, decide para cada uno: **ORIGINAL · COPIA · DERIVADO · MODIFICADO**.
Pistas: los grupos de `comparar_hashes` y el software y las dimensiones de `ver_metadatos`.

In [ ]:
ir_a("02_INSPECTORAPP")
import shutil
shutil.copy("captura_original.jpg", "mi_copia.jpg")                 # una copia exacta
if not os.path.exists("ARCHIVO_DE_MUESTRA - copia.txt"):         # la copia con un punto de más (la del mini-reto de FIJAR)
    shutil.copy("ARCHIVO_DE_MUESTRA.txt", "ARCHIVO_DE_MUESTRA - copia.txt")
    with open("ARCHIVO_DE_MUESTRA - copia.txt", "a", encoding="utf-8") as f:
        f.write(".")
!python comparar_hashes.py captura_original.jpg mi_copia.jpg captura_reducida.jpg "ARCHIVO_DE_MUESTRA - copia.txt"
!python ver_metadatos.py captura_original.jpg captura_reducida.jpg

**¿Cuál conservarías como original? ¿Cuál sellarías? ¿Cuál pondrías en un informe?**

## Una opción: un manifiesto

Una forma de documentar es un **manifiesto**: un archivo que lista, junto al original y al sello, qué es cada cosa,
de dónde sale cada dato y qué se comprobó. Es una opción, no un formato obligatorio.

## Práctica · abre un manifiesto real (paquete Orion B)

Busca estos datos y **observa de qué bloque procede cada uno**:
☐ la huella SHA-256 · ☐ la hora del sello · ☐ la huella sellada · ☐ la TSA que se llamó · ☐ la fuente / URL · ☐ si la huella sellada corresponde

Bloques: **calculado por la herramienta · obtenido del sello · configuración del proceso · aportado como contexto**.

In [ ]:
ir_a("03_PAQUETE_VERIFICACION/EVIDENCIA_orion3_paquete_B")
!cat manifiesto.json

Fíjate en la diferencia entre lo que **calculó** la herramienta, lo que **dice el sello** y lo que **alguien aportó** (la fuente, las observaciones):
lo aportado como contexto **no está verificado** por el sellado.

---
# H · VERIFICAR

Ahora toca comprobar dos cosas: **¿está bien hecho?** y **¿sirve para lo que queríamos?**

## Mini-reto · Dos paquetes técnicamente correctos

Encargo: *queríamos acreditar que la cafetera Orion 3 se ofrecía a 49,90 €.*

| | huella | sello | manifiesto |
|---|---|---|---|
| **Paquete A** | ✓ | ✓ | ✓ |
| **Paquete B** | ✓ | ✓ | ✓ |

**Los dos están técnicamente bien. ¿Los dos sirven?** Mira la captura de cada uno y decide.

In [ ]:
ir_a("03_PAQUETE_VERIFICACION")
for n in "AB":
    print("PAQUETE", n)
    display(VerImagen(f"EVIDENCIA_orion3_paquete_{n}/original.jpg", width=640))

**VALIDEZ TÉCNICA ≠ SUFICIENCIA.** Un paquete puede estar perfecto por dentro y no acreditar lo que necesitábamos.
Cuando no es suficiente, **volvemos a RECOGER**.

## Práctica · comprobamos un paquete preparado

`COMPROBAR_PAQUETE` revisa: original intacto · huella · petición · sello · correspondencia · manifiesto.
*(En clase: arrastrar la carpeta del paquete sobre `COMPROBAR_PAQUETE.bat`.)*

In [ ]:
ir_a("03_PAQUETE_VERIFICACION")
!python comprobar_paquete.py EVIDENCIA_orion3_paquete_B

**PRODUCIR ≠ VERIFICAR**: quien produce el paquete no se certifica a sí mismo. Esta herramienta es un control **independiente**.
Pero ojo: no comprueba la firma ni la cadena de la TSA (eso lo hace `/verificador`).

## Práctica · el mismo sello, otro archivo

**Prueba negativa.** Hacemos una **copia** del paquete, modificamos mínimamente (un byte) su `original.jpg` y la comprobamos con **el mismo sello**.
*Antes de ejecutar, predice: ¿qué cambiará? ¿El sello? ¿Su firma? ¿La huella del archivo? ¿La correspondencia?*

In [ ]:
ir_a("03_PAQUETE_VERIFICACION")
import shutil
shutil.rmtree("neg_B", ignore_errors=True)
shutil.copytree("EVIDENCIA_orion3_paquete_B", "neg_B")            # trabajamos sobre una COPIA del paquete
with open("neg_B/original.jpg", "ab") as f:
    f.write(b"\0")                                                # un byte más
!python comprobar_paquete.py neg_B

**MISMO SELLO + OTRO ARCHIVO → NO CORRESPONDE.** Así sabemos que el control discrimina: el sello no vale para cualquier archivo.
*(El original del paquete B no se ha tocado: el cambio se hizo sobre `neg_B`.)*

## Ruta completa

**RECOGER ✓ · CONTEXTUALIZAR ✓ · FIJAR ✓ · VINCULAR AL TIEMPO ✓ · DOCUMENTAR ✓ · VERIFICAR ✓**

---
# I · CONSUMOKU

**Diez inspectores. Un encargo. Una ganadora.** Sois el jurado. Ganará la primera candidatura **válida** recibida.
Ahora no construís: **verificáis** con lo que ya sabéis.

## El encargo y el método

El docente te dirá dónde descargar `CONSUMOKU_ALUMNADO_FINAL.zip` al empezar el reto (en clase: se descomprime dentro de `04_CONSUMOKU`).
Contiene el caso y las reglas, el manifiesto anónimo, las diez imágenes, los sellos y la hoja del jurado.

> **ADAPTADA.** Descarga el ZIP en tu ordenador y súbelo aquí con la celda siguiente
> (o arrástralo al panel 📁 de la izquierda, a la carpeta de trabajo).

**El método del reto: un embudo.**
**RELACIONA → DESCARTA RÁPIDO → REVISA EL ARCHIVO → VERIFICA SOLO LAS SUPERVIVIENTES → DECIDE.**
No verifiques sellos de candidaturas que ya has descartado por algo más barato.

In [ ]:
import zipfile, shutil
from pathlib import Path
nombre_zip = "CONSUMOKU_ALUMNADO_FINAL.zip"
candidatos = [Path.cwd() / nombre_zip, TRABAJO / nombre_zip, BASE / "04_CONSUMOKU" / nombre_zip]
zip_consumoku = next((c for c in candidatos if c.exists()), None)
if zip_consumoku is None:
    try:
        from google.colab import files
        print("Elige CONSUMOKU_ALUMNADO_FINAL.zip en tu ordenador...")
        files.upload()
        zip_consumoku = Path.cwd() / nombre_zip
    except ImportError:
        print("No se encuentra el ZIP: colócalo en la carpeta de trabajo y vuelve a ejecutar la celda.")
if zip_consumoku and zip_consumoku.exists():
    destino = BASE / "04_CONSUMOKU"
    with zipfile.ZipFile(zip_consumoku) as z:
        z.extractall(destino)
    print("Material descomprimido en:", destino.relative_to(TRABAJO))
    for p in sorted(destino.iterdir()):
        print("  ", p.name)

## El encargo · léelo con atención

Abre `00_CASO_Y_REGLAS_FINAL.pdf` (panel 📁 → `BLOQUE4/04_CONSUMOKU`, o descárgalo) y anota: **qué se buscaba, dónde, cuándo, con qué canal y con qué aplicación**.

In [ ]:
ir_a("04_CONSUMOKU")
descargar("00_CASO_Y_REGLAS_FINAL.pdf")

## Un ejemplo didáctico antes de empezar

Examinar una candidatura es **comparar lo entregado con lo exigido**, campo a campo: producto, tienda y URL, fecha y hora, canal, aplicación, sello.
Cualquier campo que no coincida la descarta. La ganadora debe cumplir **todo**.

## Trabajo del jurado

Trabajad por parejas. Las herramientas son las mismas que habéis usado: metadatos, huellas, sellos y las apps web del curso.

### 1 · El manifiesto anónimo

In [ ]:
ir_a("04_CONSUMOKU")
ver_excel("01_MANIFIESTO_ANONIMO_FINAL.xlsx")

### 2 · La hoja del jurado

In [ ]:
ver_excel("04_HOJA_DEL_JURADO.xlsx")

### 3 · Las imágenes y sus metadatos

In [ ]:
ir_a("04_CONSUMOKU")
!python ../02_INSPECTORAPP/ver_metadatos.py 02_IMAGENES_ANONIMAS

In [ ]:
# Cambia el número para ver otra imagen
display(VerImagen("02_IMAGENES_ANONIMAS/archivo_01.jpg", width=720))

### 4 · Huellas de todas las imágenes

In [ ]:
ir_a("04_CONSUMOKU")
!python ../02_INSPECTORAPP/comparar_hashes.py 02_IMAGENES_ANONIMAS

### 5 · Verificar un sello (solo para las supervivientes)

Elige el archivo y su sello y **cambia los dos nombres**. El ejemplo solo muestra cómo se usa:
**qué sello acompaña a cada archivo lo tienes que establecer tú** con el manifiesto, no lo da el ejemplo.

In [ ]:
ir_a("04_CONSUMOKU")
ARCHIVO_A_VERIFICAR = "02_IMAGENES_ANONIMAS/archivo_01.jpg"
SELLO = "03_SELLOS/INS-01_sello.tsr"
!python ../02_INSPECTORAPP/verificar_sello.py "{ARCHIVO_A_VERIFICAR}" "{SELLO}"

También puedes usar `/verificador`, `/hash` y `/exif` en la web del curso (descargando antes los archivos con `descargar(...)`).

### Tu fallo del jurado

Anota: **la ganadora**, la **segunda válida** si la hay, y las **descartadas con su motivo en una frase**.

---
# Resolución y cierre

La resolución se hace **en común**: una candidatura tras otra, con el motivo y la decisión. *(No hay nada que ejecutar.)*

## Lo que hoy hemos aprendido a no confundir

- **METADATO ≠ DATO CERTIFICADO**
- **COMPATIBLE ≠ DEMOSTRADO**
- **IP OBSERVADA ≠ PERSONA IDENTIFICADA**
- **INTEGRIDAD ≠ VERACIDAD**
- **SELLO DE TIEMPO ≠ FECHA DE CREACIÓN**
- **PRODUCIR ≠ VERIFICAR**
- **VALIDEZ TÉCNICA ≠ SUFICIENCIA**
- **VERIFICAR EL MECANISMO ≠ DEMOSTRAR EL HECHO**

*El bloque no sustituye un protocolo formal ni la valoración jurídica de una prueba.*

**Fuentes:** Reglamento (UE) 910/2014 (eIDAS) · RFC 3161 · FIPS 180-4.

✅ **Bloque 4 completado.** Sé qué recoger.

---
# 🎁 BOLA EXTRA (opcional)

**No hace falta para entender el bloque.** Se hace si sobra tiempo, o en casa.

Un sellador que **funciona**, pero es incómodo (solo terminal, sin ventana). ¿Podemos pedirle a la IA que lo convierta en una herramienta
a nuestra medida **sin tocar la parte que ya funciona** (`sellado_core.py`)?

> **ADAPTADA.** En Colab no hay ventanas: aquí ejecutas `SELLADOR_BASE.py` por terminal y preparas el prompt.
> La mejora con ventana la pruebas en tu ordenador (`EJECUTAR_SELLADOR_BASE.bat`).
> Este sellado se hace desde servidores de Google: **no tiene valor probatorio**, es solo para practicar.

## 1 · Ejecuta el sellador base con la muestra

In [ ]:
ir_a("05_BOLA_EXTRA_SELLADOR")
!python SELLADOR_BASE.py MUESTRA/captura_orion3.jpg MIS_PAQUETES https://hogarprueba.test/productos/orion-3

Si la TSA no responde desde la red de Google, no pasa nada: el programa no crea un paquete a medias. Puedes seguir con el prompt.

## 2 · El prompt: lo que quieres mejorar y lo que no se toca

In [ ]:
!cat PROMPT_BASE_SELLADOR.txt

Decide tus mejoras (sección *MIS MEJORAS*), copia el prompt en tu IA, adjunta `SELLADOR_BASE.py` y `sellado_core.py`
y guarda la respuesta como `MI_SELLADOR.py`.

## 3 · ¿Cómo sabemos que está bien?

1. `COMPROBAR_PAQUETE`: el paquete es coherente.
2. **Prueba de modificación**: una copia cambiada + el mismo sello → NO CORRESPONDE.
3. `SELLADOR_BASE.py` sigue funcionando.

In [ ]:
ir_a("03_PAQUETE_VERIFICACION")
p = ultimo_paquete()
if p:
    r = subprocess.run([sys.executable, "comprobar_paquete.py", str(p)], capture_output=True, text=True)
    print(r.stdout)
else:
    print("Todavía no hay paquete creado por tu sellador (la TSA puede no haber respondido).")

*Os lo podéis llevar y adaptar: es una base, no un producto terminado.*

---
# Solo si te has bloqueado

**Restaurar los materiales originales** (por ejemplo, si has estropeado un script).
Atención: borra también tus paquetes, tus copias y el material de CONSUMOKU que hayas descomprimido.

In [ ]:
import shutil
shutil.rmtree(BASE, ignore_errors=True)
with zipfile.ZipFile(TRABAJO / "materiales_b4.zip") as z:
    z.extractall(TRABAJO)
print("Materiales restaurados.")
ir_a("")